# Hafta 3 · Hoca demosu: API, hafıza ve JSON

Bölüm 1 ve 3'teki kod slaytlarını canlı çalıştırmak için. Açılıştaki canlı deneme sohbet uygulamasında yapılır; buradaki hücre yedektir.

**Kullanım:** Colab'da açın ([bağlantı](https://colab.research.google.com/github/asimsinan/istem-muhendisligi-dersi/blob/main/hafta-03/demo.ipynb)), kurulum hücrelerini derse girmeden çalıştırın. Ders sırasında ilgili slaytın bölümüne gidip hücreyi çalıştırın. Toplam yaklaşık **12 istek** (ücretsiz günlük sınır 50; lab'larla aynı anahtar).

Model cevapları her çalıştırmada değişebilir. **Ders öncesi bir kez baştan sona çalıştırın**; beklenen fark çıkmazsa slayt notundaki “çıkmazsa” cümlesini kullanın.

## Kurulum

Sol paneldeki 🔑 **Secrets**'ta `OPENROUTER_API_KEY` olsun ve *Notebook access* açık olsun. Bu iki hücreyi derse girmeden çalıştırın; ders sırasında yalnızca ilgili slaytın hücresini çalıştırırsınız.

In [ ]:
!pip -q install openai pydantic

In [ ]:
import json, re, time
from openai import OpenAI, RateLimitError, APIStatusError
from google.colab import userdata

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))
MODEL = "nvidia/nemotron-3.5-super-vl:free"   # yapılandırılmış çıktıyı destekler
MODEL_2 = "google/gemma-4-31b-it:free"            # strict şemayı desteklemez
BEKLE = 3                                         # dakikada 20 istek sınırı


def iste(mesajlar, model=MODEL, sicaklik=0, **ek):
    """Tek istek; 429 gelirse 30 sn bekleyip yeniden dener. Cevap nesnesini döndürür."""
    govde = dict(ek.pop("extra_body", {}))
    govde.setdefault("reasoning", {"effort": "low"})
    for deneme in range(4):
        try:
            c = client.chat.completions.create(model=model, messages=mesajlar, temperature=sicaklik,
                                               max_tokens=2000, extra_body=govde, **ek)
            time.sleep(BEKLE)
            return c
        except RateLimitError:
            print("429 · 30 sn bekleniyor…")
            time.sleep(30)
    raise RuntimeError("Üst üste 429: günlük sınır dolmuş olabilir.")


def metin(c):
    return (c.choices[0].message.content or "").strip()


def json_bul(m):
    """Cevaptaki ilk {...} bloğunu okumayı dener; okuyamazsa None."""
    b = re.search(r"\{.*\}", m or "", re.S)
    try:
        return json.loads(b.group(0)) if b else None
    except json.JSONDecodeError:
        return None

print("Hazır.")

SISTEM = '# Rol\nMüşteri yorumlarını puanlayan bir analistsin.\n\n# Bağlam\nYorumlar bir elektronik mağazasından; ironi sık görülür.\n\n# Çıktı biçimi\nJSON, üç alan: gerekce (kısa metin), duygu (olumlu | olumsuz | karışık), puan (1–5).\n\n# Kısıtlar ve üslup\nGerekçe en fazla 15 kelime, Türkçe. Emin değilsen "karışık" de.'
GOREV = "Bu yorumun duygusunu ve 1–5 puanını belirle."
YORUM = "Telefon harika ama şarj aleti iki günde bozuldu."


def mesajlar(sistem, yorum):
    return [{"role": "system", "content": sistem},
            {"role": "user", "content": f"{GOREV}\n<yorum>\n{yorum}\n</yorum>"}]

## H3·4 · Canlı deneme (yedek)

Sohbet uygulaması açılmazsa: aynı cümle, sistem mesajı yok, şema yok. Cevap büyük olasılıkla birkaç paragraf ve gömülü bir puan.

In [ ]:
c = iste([{"role": "user", "content": f"Bu yorumu 1 ile 5 arasında puanla: {YORUM}"}])
print(metin(c))

## H3·18–19 · Python'dan ilk istek ve cevabın anatomisi

Slayttaki kodun aynısı (sistem = `prompts/v1.txt`). İkinci hücre cevabın tamamını JSON olarak gösterir: `content`, `finish_reason`, `usage`. Slayttaki değerler örnekti; bunlar gerçek.

In [ ]:
cevap = iste(mesajlar(SISTEM, YORUM), model=MODEL_2)
print(cevap.choices[0].message.content)

In [ ]:
d = cevap.model_dump()
ozet = {"id": d["id"], "model": d["model"],
        "choices": [{"message": {"role": d["choices"][0]["message"]["role"],
                                 "content": d["choices"][0]["message"]["content"]},
                     "finish_reason": d["choices"][0]["finish_reason"]}],
        "usage": {k: d["usage"][k] for k in ("prompt_tokens", "completion_tokens", "total_tokens")}}
print(json.dumps(ozet, ensure_ascii=False, indent=2))

## H3·16–17 · API hafızasızdır

Üç istek: (1) adımızı söyleriz, (2) geçmiş **olmadan** adımızı sorarız, (3) geçmişle birlikte sorarız. Girdi token'ının her turda nasıl büyüdüğüne bakın.

In [ ]:
tur1 = [{"role": "user", "content": "Merhaba, benim adım Deniz. Kısaca selam ver."}]
c1 = iste(tur1)
print("1. tur  :", metin(c1)[:100], f"   (girdi {c1.usage.prompt_tokens} token)")

soru = {"role": "user", "content": "Benim adım ne?"}
c2 = iste([soru])
print("Geçmişsiz:", metin(c2)[:100], f"   (girdi {c2.usage.prompt_tokens} token)")

gecmisli = tur1 + [{"role": "assistant", "content": metin(c1)}, soru]
c3 = iste(gecmisli)
print("Geçmişle :", metin(c3)[:100], f"   (girdi {c3.usage.prompt_tokens} token)")

## H3·33 · “JSON ver” demek, JSON almak demek değil

Şemasız, `prompts/v1.txt` ile üç yorum. Ham cevabı `repr` ile gösterir: kod çiti, baştaki cümle, fazladan alan. `json.loads` doğrudan okuyabiliyor mu?

In [ ]:
UC_YORUM = ["Harika! İkinci gün açılmaz oldu, tam da beklediğim gibi.",
            "Ekranı çok güzel ama pil yarım günü zor çıkarıyor.",
            "Kargo ertesi gün geldi, ürün tam anlatıldığı gibi."]
okunamayan = 0
for y in UC_YORUM:
    m = metin(iste(mesajlar(SISTEM, y), model=MODEL_2))
    try:
        json.loads(m)
        durum = "✓ json.loads okudu"
    except json.JSONDecodeError:
        okunamayan += 1
        durum = "✕ json.loads okuyamadı"
    print(durum, "→", repr(m[:140]))
print(f"\nDoğrudan okunamayan: {okunamayan}/{len(UC_YORUM)}")

## H3·38 · Şema ve Pydantic

Aynı üç yorum, slayttaki istek: `json_schema` + `strict` + `require_parameters`, Pydantic ile doğrulama. Geçersiz sayısı 0 olmalı.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, ValidationError


class Puanlama(BaseModel):
    model_config = ConfigDict(extra="forbid")
    gerekce: str
    duygu: Literal["olumlu", "olumsuz", "karışık"]
    puan: Literal[1, 2, 3, 4, 5]


SEMA = Puanlama.model_json_schema()
SEMALI = {"response_format": {"type": "json_schema", "json_schema": {"name": "puanlama", "strict": True, "schema": SEMA}},
          "extra_body": {"provider": {"require_parameters": True}}}

gecersiz = 0
for y in UC_YORUM:
    m = metin(iste(mesajlar(SISTEM, y), **SEMALI))
    try:
        p = Puanlama.model_validate_json(m)
        print(f"✓ {p.duygu:8} {p.puan}  ← {y[:50]}")
    except ValidationError as e:
        gecersiz += 1
        print("✕", e.errors()[0]["msg"], "→", m[:80])
print(f"\nGeçersiz: {gecersiz}/{len(UC_YORUM)}")

## H3·64 · Bugünün sorusuna dönüş

Açılıştaki yorum, şimdi sistem istemi ve şemayla: üç paragraf yerine tek bir JSON.

In [ ]:
print(metin(iste(mesajlar(SISTEM, YORUM), **SEMALI)))